In [ ]:
import torch

from rlaopt.atoms import L1Norm, SumSquares
from rlaopt.expression import Variable
from rlaopt.solvers import ProxGrad, ProxGradConfig, ProxGradStoppingCriteria

In [ ]:
torch.set_default_dtype(torch.float32)
# torch.set_default_dtype(torch.float64)

In [ ]:
# A = torch.tensor([[3.0, 2.0], [2.0, 4.0]])
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([5.0, 11.0])
w = Variable((2, 2), name="w")
x = Variable(torch.tensor([1.0, 2.0]), name="x")
y = Variable(torch.tensor([3.0, 4.0]), name="y")
z = Variable(torch.tensor([1.0, 2.0]), name="z")

loss = 2 * (
    SumSquares(A @ x - b)
    + SumSquares(A @ y - b)
    + SumSquares(x + y)
    + 2 * L1Norm(y, scaling=2.0)
    + L1Norm(x)
    + L1Norm(z)
)

# loss = L1Norm(x)
loss = loss.cuda()

# loss = 2 * (SumSquares(A @ x - b) + SumSquares(A @ y - b) + 2 * L1Norm(y, scaling=2.0) + L1Norm(x) + L1Norm(z))

# loss = 2 * (SumSquares(A @ x - b) + SumSquares(A @ y - b) +
#             SumSquares(x + y) + 2 * L1Norm(y, scaling=2.0) + L1Norm(x))

# def true_loss(x, y, z):
#     return 2 * (torch.sum((A @ x - b) ** 2) + torch.sum((A @ y - b) ** 2) + \
#         torch.sum((x + y) ** 2) + 4 * torch.sum(torch.abs(y)) + torch.sum(torch.abs(x)) + torch.sum(torch.abs(z)))

In [ ]:
loss.forward()

### Try out expression tree

In [ ]:
print(loss.tree())

In [ ]:
loss_1 = A @ (A @ x)
print(loss_1.tree())

In [ ]:
loss_2 = SumSquares(A @ x - b) + L1Norm(x)
loss_3 = L1Norm(x) + SumSquares(A @ x - b)
print(loss_2.tree())
print(loss_3.tree())

In [ ]:
loss_2.tree() == loss_3.tree()

In [ ]:
loss_4 = 0.25 * (x.T @ A @ x) * 2 + b.T @ x
print(loss_4.tree())

In [ ]:
loss_5 = w.T
print(loss_5.tree())

In [ ]:
loss_6 = w.sum(dim=0)
print(loss_6.tree())

In [ ]:
loss_7 = (A @ x - b) ** 2
print(loss_7.tree())

### Optimize objective

In [ ]:
obj = loss

In [ ]:
solver = ProxGrad(
    obj, ProxGradConfig(eta=1.0, use_acceleration=False, use_linesearch=True)
)
num_iterations = 1000
stopping_criteria = ProxGradStoppingCriteria(tol=1e-6, max_iters=num_iterations)

In [ ]:
params = obj.variable_values
state = solver.init_state(params)
for i in range(num_iterations):
    params, state = solver.step(params, state)
    if (i + 1) % 20 == 0:
        print(f"Iteration {i + 1}: error: {state.err}, obj: {obj.evaluate(params)}")

In [ ]:
result = solver.solve(stopping_criteria=stopping_criteria)
print(
    f"Solved params: {[f'{name}: {value}' for name, value in result.variable_values.items()]}, convergence_status: {result.convergence_status}, final error: {result.err}"
)